# Carga y limpieza de transacciones retail

Pipeline inicial del proyecto de **optimización de inventario e-commerce**.

1. Cargar Online Retail II (UCI / Kaggle) o el sample local
2. Gestionar nulos y registros inválidos
3. Calcular **ventas totales por producto** del **último trimestre**

> Dataset completo: `data/raw/online_retail_II.csv` · Sample: `data/raw/sample_online_retail.csv`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "inventario_ecommerce").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions, save_processed
from inventario_ecommerce.features import clean_transactions, sales_by_product_last_quarter
from inventario_ecommerce.plots import plot_top_products_by_sales

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", config.RAW_DATA_DIR)

## 1. Carga del dataset

Fuente activa: **`data/raw/online_retail_II.csv`** (Online Retail II UCI / Kaggle, ~1.07M filas).

El sample `sample_online_retail.csv` queda solo para smoke-tests rápidos.

In [ ]:
RAW_FILE = config.DEFAULT_RAW_FILE  # data/raw/online_retail_II.csv
# Smoke-test: RAW_FILE = config.RAW_DATA_DIR / "sample_online_retail.csv"

df_raw = load_transactions(RAW_FILE)
print(f"Archivo: {RAW_FILE.name}")
print(f"Filas: {len(df_raw):,} | Columnas: {list(df_raw.columns)}")
df_raw.head()

## 2. Diagnóstico de calidad de datos

In [ ]:
print("=== Info ===")
df_raw.info()

print("\n=== Nulos por columna ===")
nulls = df_raw.isna().sum().sort_values(ascending=False)
null_pct = (df_raw.isna().mean() * 100).round(2)
quality = pd.DataFrame({"nulos": nulls, "%": null_pct})
display(quality[quality["nulos"] > 0])

print("\n=== Duplicados exactos ===")
print(df_raw.duplicated().sum())

## 3. Limpieza

Reglas aplicadas:
- Eliminar filas con `StockCode`, `Description`, `InvoiceDate` o `Price` nulos
- Descartar cantidades <= 0 (devoluciones) y precios negativos
- Crear columna `Sales = Quantity * Price`

In [ ]:
df_clean = clean_transactions(df_raw)

print(f"Filas raw:     {len(df_raw):,}")
print(f"Filas limpias: {len(df_clean):,}")
print(f"Filas removidas: {len(df_raw) - len(df_clean):,}")
print(f"Rango de fechas: {df_clean['InvoiceDate'].min()} -> {df_clean['InvoiceDate'].max()}")
print(f"Nulos residuales: {int(df_clean.isna().sum().sum())}")
df_clean.head()

## 4. Ventas totales por producto — último trimestre

Se toma como referencia la fecha máxima del dataset y se agregan los 3 meses previos.

In [ ]:
sales_q = sales_by_product_last_quarter(df_clean)

period_start = sales_q.attrs.get("period_start")
period_end = sales_q.attrs.get("period_end")
print(f"Periodo analizado: {period_start.date()} -> {period_end.date()}")
print(f"Productos con ventas: {len(sales_q):,}")
print(f"Ventas totales del trimestre: {sales_q['TotalSales'].sum():,.2f}")

sales_q.head(15)

In [ ]:
out_path = save_processed(sales_q, "sales_last_quarter_by_product.csv")
print(f"Guardado en: {out_path}")

plot_top_products_by_sales(sales_q, top_n=10, save=True);

## Ver también

- `notebooks/02_forecast_reorder_baseline.ipynb` — ABC, forecast baseline y punto de reorden
- `notebooks/03_demand_hygiene.ipynb` — calendario con ceros, winsor y filtros
- `references/data_dictionary.md` — columnas y reglas de limpieza